<a href="https://colab.research.google.com/github/Gnissan-BIA/Assignments/blob/main/Project_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Advanced Data Cleaning

Import dataset

In [44]:
import pandas as pd
import numpy as np
import re

#load dataset

df=pd.read_csv('merged_dataset.csv')
print(df.head())
print()
df.info()

  ticker   open  close  adj_close    low   high   volume  year  decade  \
0    AHH  11.50  11.58   8.493155  11.25  11.68  4633900  2013    2010   
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800  2013    2010   
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100  2013    2010   
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400  2013    2010   
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100  2013    2010   

  exchange                             name   sector     industry  
0     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE  
1     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE  
2     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE  
3     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE  
4     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180233 entries, 0 to 180232
Data columns (total 13 columns):
 #   Column     Non-Null Count   Dt

Handle Missing Values

In [45]:
print(df.isnull().sum())

ticker       0
open         0
close        0
adj_close    0
low          0
high         0
volume       0
year         0
decade       0
exchange     0
name         0
sector       0
industry     0
dtype: int64


Data was cleaned in project part 1 so has no missing values.

In [ ]:

#Other imputation methods
# # Forward fill (good for time series)
# df = df.sort_index()
# df[['open','high','low','close']] = df[['open','high','low','close']].ffill()

# # Backward fill (optional)
# df[['open','high','low','close']] = df[['open','high','low','close']].bfill()

# # Interpolation (smooth estimation)
# df[['open','high','low','close']] = df[['open','high','low','close']].interpolate(method='linear')

# # Volume (use median - less sensitive to outliers)
# df['volume'] = df['volume'].fillna(df['volume'].median())


Detect & Handle Outliers (IQR Method)

In [46]:
def cap_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[column] = np.where(df[column] < lower, lower,
                  np.where(df[column] > upper, upper, df[column]))
    return df

for col in ['close', 'volume']:
    df = cap_outliers(df, col)

#Error Detection & Correction

In [47]:

#Check for unrealistic values:
print(df.describe())

# Remove negative prices
df = df[(df['close'] > 0) & (df['open'] > 0)]

# Ensure High >= Low
df = df[df['high'] >= df['low']]


                open          close      adj_close            low  \
count  180233.000000  180233.000000  180233.000000  180233.000000   
mean       18.064329      15.332955      14.279486      17.765564   
std        27.701472      12.101300      27.081437      26.874818   
min         0.050000       0.123261       0.000015       0.015000   
25%         6.260000       6.260000       3.542049       6.130000   
50%        12.500000      12.500000       8.140000      12.280000   
75%        21.170000      21.170000      16.347357      20.820000   
max       686.006836      43.535000     666.808899     643.913513   

                high        volume           year         decade  
count  180233.000000  1.802330e+05  180233.000000  180233.000000  
mean       18.351945  3.269104e+05    2005.618361    2000.915759  
std        28.436212  4.191041e+05       9.514261       9.599588  
min         0.125867  1.000000e+02    1973.000000    1970.000000  
25%         6.390000  2.300000e+04    2000.

#Data Transformation

Moving Average

In [48]:
df['MA_10'] = df['close'].rolling(window=10).mean()
df['MA_50'] = df['close'].rolling(window=50).mean()

print(df.head())

  ticker   open  close  adj_close    low   high     volume  year  decade  \
0    AHH  11.50  11.58   8.493155  11.25  11.68  1174500.0  2013    2010   
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800.0  2013    2010   
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100.0  2013    2010   
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400.0  2013    2010   
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100.0  2013    2010   

  exchange                             name   sector     industry  MA_10  \
0     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   
1     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   
2     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   
3     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   
4     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   

   MA_50  
0    NaN  
1    NaN  
2    NaN  
3    NaN  
4    NaN  


Volatility

In [49]:
df['volatility'] = df['close'].rolling(window=10).std()

Daily Returns

In [50]:
df['returns'] = df['close'].pct_change()

RSI

In [51]:
delta = df['close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()

rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

In [52]:
print(df.head())

  ticker   open  close  adj_close    low   high     volume  year  decade  \
0    AHH  11.50  11.58   8.493155  11.25  11.68  1174500.0  2013    2010   
1    AHH  11.66  11.55   8.471151  11.50  11.66   275800.0  2013    2010   
2    AHH  11.55  11.60   8.507822  11.50  11.60   277100.0  2013    2010   
3    AHH  11.63  11.65   8.544494  11.55  11.65   147400.0  2013    2010   
4    AHH  11.60  11.53   8.456484  11.50  11.60   184100.0  2013    2010   

  exchange                             name   sector     industry  MA_10  \
0     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   
1     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   
2     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   
3     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   
4     NYSE  ARMADA HOFFLER PROPERTIES, INC.  FINANCE  REAL ESTATE    NaN   

   MA_50  volatility   returns  RSI  
0    NaN         NaN       NaN  NaN  
1    NaN  

Normalization/ Standardization

In [53]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[['open','high','low','close','volume']] = scaler.fit_transform(
    df[['open','high','low','close','volume']]
)


In [61]:
print(df.head())

   ticker      open     close  adj_close       low      high    volume  year  \
49    AHH -0.250324 -0.339878   8.287378 -0.249884 -0.247289 -0.384179  2013   
50    AHH -0.246353 -0.335746   8.324308 -0.249884 -0.248344 -0.579357  2013   
51    AHH -0.245992 -0.341531   8.272604 -0.244675 -0.248344 -0.662153  2013   
52    AHH -0.247797 -0.339052   8.294762 -0.246163 -0.246937 -0.621590  2013   
53    AHH -0.246353 -0.343184   8.257831 -0.250628 -0.250102 -0.585084  2013   

    decade exchange  ...        RSI sector_CONSUMER DURABLES  \
49    2010     NYSE  ...  44.285721                    False   
50    2010     NYSE  ...  36.065600                    False   
51    2010     NYSE  ...  36.871515                    False   
52    2010     NYSE  ...  36.516852                    False   
53    2010     NYSE  ...  34.444469                    False   

    sector_CONSUMER NON-DURABLES  sector_CONSUMER SERVICES  sector_ENERGY  \
49                         False                     Fals

Encoding Categorical Variables

In [ ]:
# Sector column
df = pd.get_dummies(df, columns=['sector'], drop_first=True)

#Integration and Formatting for Modeling

Handle Missing values

In [56]:
if df is None:
    print("Error: DataFrame 'df' is None. Please ensure previous steps correctly initialized 'df' as a pandas DataFrame.")
else:
    df = df.dropna()
    print(df.tail())

       ticker      open     close  adj_close       low      high    volume  \
180228    RJF -0.532670 -0.993641   1.504255 -0.539775 -0.529021 -0.661915   
180229   MCHX -0.418908 -0.728267   5.064775 -0.421048 -0.413275 -0.552395   
180230    MHI -0.247436 -0.342357  10.933822 -0.245047 -0.250102 -0.694842   
180231   FLOW -0.458075 -0.822885   5.375000 -0.461049 -0.447562 -0.769525   
180232    MHI -0.249963 -0.348142  10.865425 -0.247279 -0.252212 -0.641633   

        year  decade exchange  ...        RSI sector_CONSUMER DURABLES  \
180228  1993    1990     NYSE  ...  50.000000                    False   
180229  2010    2010   NASDAQ  ...  48.011009                    False   
180230  2018    2010     NYSE  ...  48.785607                    False   
180231  1987    1980     NYSE  ...  49.463263                    False   
180232  2018    2010     NYSE  ...  40.054169                    False   

        sector_CONSUMER NON-DURABLES  sector_CONSUMER SERVICES  sector_ENERGY  \
18022

Data Splitting

In [64]:
#Time Series Split (Important)
train = df[df['year'] <= 2015]
val = df[(df['year'] >= 2016) & (df['year'] <= 2018)]
test = df[df['year'] >= 2019]

#Alternative (sklearn)
from sklearn.model_selection import train_test_split

X = df.drop('close', axis=1) # Corrected 'Close' to 'close'
y = df['close'] # Corrected 'Close' to 'close'

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False)


Save & download trained files for future use

In [66]:
import pickle
from google.colab import files

data = {
    "train": train,
    "val": val,
    "test": test,
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
}

with open("data_splits.pkl", "wb") as f:
    pickle.dump(data, f)

files.download("data_splits.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>